## Session 7, in class: from one household to many

Computational Macro (HWS 2026), University of Mannheim. Thursday, 22.10.2026, 10:00 to 12:00. Not graded.

This is the code behind the session 6 slides. Part 1 solves the household of the slides with the endogenous
grid method of week 5. Part 2 aggregates by Monte Carlo simulation, part 3 by Young's method, with a close
look at where the mass of one cell goes. Profiling the two, and one equilibrium wrapper for Huggett and
Aiyagari, follow in the coding session after the midterm (`Week_8`).

**How to work with this notebook**

- Run the cells from top to bottom (`Shift + Enter`). After a restart of the kernel, start again at the top:
  later cells use what earlier cells define.
- There are **4 gaps**, marked `___`. The comment line `# Gap k: ...` directly above a gap says what goes in.
  Every gap is one line, and most are one line of the slides.
- Cells without a gap run as they are. Read them anyway: they are part of the method, and the comments
  explain each step.
- Each part ends with a **Check**: a number to compare with, from the slides. Do not move on before it fits.
- **Slides.** "Session 6, slide 12" and the like refer to the session 6 slides, by the number at the bottom of
  each slide. Keep them open next to the notebook.
- The solution notebook is posted after the session.

**Plan**

| Part | Content | Gaps | Slides | Time |
| --- | --- | --- | --- | --- |
| 1 | The household of the slides, solved with the endogenous grid method (given) | | S6: 2 | 5 min |
| 2 | Aggregation by Monte Carlo simulation: a panel of households, and the sampling error | | S6: 6 to 10 | 10 min |
| 3 | Young's method: the lottery, where the mass of one cell goes, the policy chain, the stationary distribution | 1 to 4 | S6: 11 to 17 | 30 min |

**Notation, slides to code**

| Slides | Meaning | Code |
| --- | --- | --- |
| $\mathcal{A}$, $\mathcal{Z}$, $\boldsymbol{\Pi}$ | asset grid, income grid, transition matrix (rows sum to one) | `gri.a`, `gri.z`, `gri.Pi` |
| $\boldsymbol{\lambda}^z$ | stationary distribution of the income chain | `gri.lam_z` |
| $\mathbf{c}$, $\mathbf{g}$, $a'_{im} = g(a_i, z_m)$ | consumption and savings policy on the grid | `C`, `G` |
| $l$, $w_{im} = (a'_{im} - a_l) / (a_{l+1} - a_l)$ | the lottery: left neighbour of $a'_{im}$ and the weight on the right one | `l`, `w` |
| $\mathbf{Q}_{\mathbf{g}}$ | the policy chain, $N_a N_z \times N_a N_z$, row $(i, m)$ is the state $(a_i, z_m)$ | `Q` |
| $\boldsymbol{\lambda}$, $\boldsymbol{\lambda} = \mathbf{Q}_{\mathbf{g}}^\top \boldsymbol{\lambda}$ | distribution over $(a, z)$, stationary | `lam` |
| $A = \boldsymbol{\lambda}^\top \mathbf{a}$ | mean assets | `mean_assets(lam, gri)` |

**Toolbox: the few tools this notebook uses that are new**

| Tool | What it does | Example |
| --- | --- | --- |
| `np.searchsorted(grid, x, side="right") - 1` | the index `l` of the grid point just below `x` (for a sorted `grid`) | locating the lottery |
| `np.clip(x, lo, hi)` | elementwise: `x`, but not below `lo` and not above `hi` | keep `l` inside `0, ..., N_a - 2` |
| `np.arange(n).reshape(nz, na).T` | the numbers `0, ..., n - 1` as an `na` by `nz` table, counting down the columns | `idx[i, m]` is the row of state $(a_i, z_m)$ in `Q` |
| `sp.csr_matrix((vals, (rows, cols)), shape=(n, n))` | a sparse matrix from triplets: `vals[k]` sits at `(rows[k], cols[k])` | $\mathbf{Q}_{\mathbf{g}}$ with $2 N_z$ entries per row |
| `Q.T @ lam` | sparse matrix times vector | one step of the distribution |
| `rng = np.random.default_rng(seed)`, `rng.random(N)` | a seeded random number generator, `N` uniforms | the simulation |

Everything else is from the earlier notebooks: `A @ B`, `x[:, None]`, `np.interp`, `A.sum(axis=1)`.
A policy is stored as an $N_a \times N_z$ array: row `i` is the asset grid point $a_i$, column `m` the income state $z_m$.

### 0. Packages

Nothing new to install: `scipy` (for the sparse matrix) and `quantecon` (for the income chain) are in the
course environment since weeks 3 and 5, `time`, `cProfile`, and `pstats` ship with Python. The first cell
installs the packages into the environment of the running kernel if they are missing.

In [ ]:
# Installs the packages of this notebook into the environment of the running kernel.
%pip install "numpy==2.2.*" "scipy==1.15.*" "matplotlib==3.10.*" "quantecon>=0.7"

In [ ]:
import time
import cProfile
import pstats
import warnings
from dataclasses import dataclass
from typing import Callable

import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import quantecon as qe

warnings.filterwarnings("ignore", message="The API of rouwenhorst")   # the notice on every rouwenhorst call, as in week 5

### 1. The household of the slides (given)

*Session 6, slides 2 and 4.*

The household of sessions 4 and 5 with CRRA utility, $u(c) = c^{1 - \gamma} / (1 - \gamma)$, $\gamma = 2$,
$\beta = 0.96$, the income chain of session 4 ($\ln z' = \rho \ln z + \sigma_\varepsilon \varepsilon'$ with
$\rho = 0.9$, $\sigma_\varepsilon = 0.2$, on $N_z = 7$ Rouwenhorst states, $z = e^{\ln z}$), and the budget
constraint $c + a' = w z + R a$ with $a' \geq \underline{a}$. In parts 2 and 3 prices are given, $r = 0.03$ and
$w = 1$, and $\underline{a} = 0$. Huggett changes $\underline{a}$, Aiyagari changes $w$, both change $r$, so
the solver takes `r`, `w`, and the grids as arguments.

The asset grid has $N_a = 200$ points on $[\underline{a}, 100]$, **quadratically spaced**: dense near the
limit, where the policy bends, sparse at the top, where it is almost linear. The household problem is solved
with the endogenous grid method of session 5 (linear interpolation, policy tolerance $10^{-8}$). The only
changes to week 5's `egm_step` are CRRA marginal utility, income $w z_m$ in cash on hand, and the persistent
chain in $\mathbf{EMU} = u_c(\mathbf{c}) \boldsymbol{\Pi}^\top$, which was there already.

In [ ]:
@dataclass(frozen=True)
class EconomicParameters:
    gamma: float = 2.0        # relative risk aversion
    beta: float = 0.96        # discount factor
    rho: float = 0.9          # persistence of ln z
    sig_eps: float = 0.2      # standard deviation of the innovation to ln z


@dataclass(frozen=True)
class NumericalParameters:
    na: int = 200             # asset grid points N_a
    nz: int = 7               # income grid points N_z
    amax: float = 100.0       # upper grid bound a_{N_a}
    crit_pol: float = 1e-8    # the EGM stops when the policy moves by less
    crit_dist: float = 1e-12  # the distribution iteration stops when lam moves by less (part 3)
    maxiter: int = 100_000    # iteration cap, the safety net

In [ ]:
@dataclass(frozen=True)
class Grids:
    a: np.ndarray             # asset grid, N_a
    z: np.ndarray             # income grid, N_z
    Pi: np.ndarray            # transition matrix, N_z x N_z
    lam_z: np.ndarray         # stationary distribution of the income chain, N_z


def make_grids(amin, par, mpar):
    # The income chain of session 4 on N_z Rouwenhorst states (week 5, part 1), income levels z = exp(ln z),
    # and the quadratically spaced asset grid: a_i = amin + (amax - amin) s_i^2 with s_i equidistant on [0, 1]
    mc = qe.rouwenhorst(mpar.nz, par.rho, par.sig_eps)   # the chain for ln z, as in week 5
    z = np.exp(mc.state_values)                          # income levels
    lam_z = mc.stationary_distributions[0]               # the stationary distribution of the chain, week 4
    s = np.linspace(0.0, 1.0, mpar.na)                   # N_a equidistant points on [0, 1]
    a = amin + (mpar.amax - amin) * s ** 2               # squared: dense near amin, sparse near amax
    return Grids(a=a, z=z, Pi=mc.P, lam_z=lam_z)

In [ ]:
par, mpar = EconomicParameters(), NumericalParameters()  # the default values listed above
gri = make_grids(0.0, par, mpar)                         # borrowing limit a >= 0 for parts 2 and 3
print("income states:", np.round(gri.z, 3))
print("stationary distribution of income:", np.round(gri.lam_z, 3), " mean income E[z] =", round(gri.lam_z @ gri.z, 4))
print("asset grid: first spacing", round(gri.a[1] - gri.a[0], 4), " last spacing", round(gri.a[-1] - gri.a[-2], 3))

**The solver.** `uprime` and `uprime_inv` are the CRRA versions of week 5's CARA functions.
`egm_step` and `solve_household` are week 5's `egm_step` and `egm` with `r`, `w`, and the grids as arguments.
The guess is to consume income and interest, $c_0 = r a + w z$, unless a better guess is passed in `C0`
(next week explains why that matters). The solver returns consumption `C`, the savings policy `G` with
$a'_{im} = g(a_i, z_m) = w z_m + R a_i - c_{im}$, and the iteration count.

In [ ]:
def uprime(c, par):
    # marginal utility of CRRA utility u(c) = c^(1 - gamma) / (1 - gamma), that is u'(c) = c^(-gamma)
    return c ** (-par.gamma)                    # works on numbers and arrays


def uprime_inv(m, par):
    # inverse of marginal utility: the c with u'(c) = m
    return m ** (-1 / par.gamma)

In [ ]:
def egm_step(C, r, w, par, gri):
    # One step of the endogenous grid method (session 5, slide 20): from the policy C to the new policy on the grid
    R = 1 + r
    EMU = uprime(C, par) @ gri.Pi.T             # expected marginal utility tomorrow at every (a'_l, z_m), week 5 gap 5
    ctilde = uprime_inv(par.beta * R * EMU, par)  # (a) consumption today from the inverted Euler equation
    atilde = (ctilde + gri.a[:, None] - w * gri.z[None, :]) / R   # (b) today's assets from the budget constraint
    Cnew = np.empty_like(C)                     # (c) back to the grid, one income state at a time
    for m in range(len(gri.z)):
        Cnew[:, m] = np.interp(gri.a, atilde[:, m], ctilde[:, m])
        below = gri.a < atilde[0, m]            # poorer than the household that saves a' = a_1: the limit binds
        Cnew[below, m] = w * gri.z[m] + R * gri.a[below] - gri.a[0]   # consume everything down to a' = a_1
    return Cnew

In [ ]:
def solve_household(r, w, par, mpar, gri, C0=None):
    # The EGM loop of week 5: iterate egm_step until the policy stops moving. Returns C, the savings policy G, and the count.
    # C0: an optional starting guess for the policy (the solution at nearby prices, next week)
    R = 1 + r
    C = np.maximum(r * gri.a[:, None] + w * gri.z[None, :], 1e-6) if C0 is None else C0.copy()   # the guess: consume interest and income
    for it in range(1, mpar.maxiter + 1):
        Cnew = egm_step(C, r, w, par, gri)
        dist = np.max(np.abs(Cnew - C))         # largest change of the policy
        C = Cnew
        if dist < mpar.crit_pol:
            break
    G = w * gri.z[None, :] + R * gri.a[:, None] - C   # savings a' = w z + R a - c, N_a x N_z
    return C, G, it

In [ ]:
r0, w0 = 0.03, 1.0                              # the prices of parts 2 and 3
t0 = time.perf_counter()
C0, G0, it0 = solve_household(r0, w0, par, mpar, gri)
print(f"EGM: {it0} iterations, {time.perf_counter() - t0:.3f} s")
print("net saving g(a, z) - a at a = 0, by income state:", np.round(G0[0] - gri.a[0], 3))
i10 = np.argmin(np.abs(gri.a - 10))             # the grid point closest to a = 10
print(f"net saving at a = {gri.a[i10]:.2f}, by income state:", np.round(G0[i10] - gri.a[i10], 3))

**Check (session 6, slide 2):** mean income $E[z] = 1.11$, EGM converges in $369$ iterations. At $a = 0$ the two
lowest income states stay at the limit (net saving $0$), the highest saves about $0.9$. At $a = 10$ every state
but the two highest dissaves. A household with median income has a target wealth where saving turns negative:
that is where mass will pile up.

In [ ]:
# Nothing to fill in: the savings policy as net saving g(a, z) - a, as on session 6, slide 2
fig, ax = plt.subplots(figsize=(6, 3.4))
sel = gri.a <= 30
for k, lab in ((0, "lowest income $z_1$"), (3, "median income $z_4$"), (6, "highest income $z_7$")):
    ax.plot(gri.a[sel], G0[sel, k] - gri.a[sel], lw=1.5, label=lab)
ax.axhline(0, color="k", lw=0.8, ls=":")
ax.set(xlabel="assets today $a$", ylabel="net saving $g(a, z) - a$", title="The savings policy")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 2. Aggregation by Monte Carlo simulation (nothing to fill in)

*Session 6, slides 6 to 10.*

The direct way to a distribution: $N$ households, each with its own assets and income, moved forward $T$
periods with the policy and the income chain. Slide 6: start everybody at $a = 0$ with income drawn from
$\boldsymbol{\lambda}^z$, and in every period (i) read tomorrow's assets off the policy, interpolated between
grid points, and (ii) draw tomorrow's income from the row of $\boldsymbol{\Pi}$ of today's income, with the
cumulated row and one uniform (week 5, part 1). **Vectorize over households, not over time:** one
interpolation per income state per period, one array of $N$ uniforms.

The cross section after $T$ periods is a sample from the stationary distribution, once the burn in is over.

In [ ]:
def simulate_panel(G, gri, N, T, seed):
    # N households for T periods. Returns the final cross section (assets, income index) and the mean of assets in every period
    rng = np.random.default_rng(seed)           # seeded, so that the run is reproducible
    cum = np.cumsum(gri.Pi, axis=1)
    a = np.full(N, gri.a[0])                    # everybody starts at the limit
    m = rng.choice(len(gri.z), size=N, p=gri.lam_z)   # income indices drawn from the stationary distribution
    means = np.empty(T)                         # mean assets in every period
    for t in range(T):
        anew = np.empty(N)
        for k in range(len(gri.z)):             # (i) tomorrow's assets: one interpolation per income state
            sel = m == k                        # the households with income z_k today
            anew[sel] = np.interp(a[sel], gri.a, G[:, k])
        a = anew
        u = rng.random(N)                       # (ii) tomorrow's income: one uniform per household
        m = np.minimum((u[:, None] > cum[m]).sum(axis=1), len(gri.z) - 1)   # the first n with F_mn >= u, for every household at once (week 5 by hand)
        means[t] = a.mean()
    return a, m, means

In [ ]:
N, T = 10_000, 1_000
t0 = time.perf_counter()
a_sim, m_sim, means = simulate_panel(G0, gri, N, T, seed=11)
t_sim = time.perf_counter() - t0
print(f"simulation of N = {N} households for T = {T} periods: {t_sim:.2f} s")
print(f"mean assets {a_sim.mean():.3f}, share at the limit {(a_sim <= gri.a[0] + 1e-10).mean():.3f}, median {np.median(a_sim):.2f}")
print("mean assets over time, t = 10, 50, 100, 300, 1000:", np.round(means[[9, 49, 99, 299, 999]], 2))

**Check (session 6, slide 10):** mean assets about $8.5$ (the seed decides the second decimal), $1$ to $2$ percent
at the limit, a median around $6$. The mean needs about $300$ periods to settle (the burn in), the first $10$
periods are still close to zero.

**The sampling error.** Repeat the simulation with three seeds for $N = 1{,}000$ and $10{,}000$ households
($T = 500$ is enough after the burn in check above). The standard deviation of the mean across seeds falls
like $1 / \sqrt{N}$: one more digit costs a hundred times the households, and the time grows with them.
(Next week the numbers go next to Young's method.)

In [ ]:
mc_rows = []                                    # (N, mean over seeds, sd across seeds, seconds per simulation), for next week
for Nn in (1_000, 10_000):
    ms, tt = [], time.perf_counter()
    for seed in range(3):
        a_s, _, _ = simulate_panel(G0, gri, Nn, 500, seed=seed)
        ms.append(a_s.mean())
    tt = (time.perf_counter() - tt) / 3
    mc_rows.append((Nn, np.mean(ms), np.std(ms), tt))
    print(f"N = {Nn:7d}: mean assets over 3 seeds {np.mean(ms):.3f}, sd across seeds {np.std(ms):.3f}, {tt:.2f} s per simulation")

**Check:** the standard deviation across seeds is about $0.2$ and $0.07$ (with three seeds the first digit
moves between runs): it falls by a factor of about $\sqrt{10} \approx 3.2$ for the factor of $10$ in $N$. To pin the mean to
two decimals you would need $N = 10^5$ and a hundred times the time. Keep these numbers: Young's method delivers
the mean to twelve digits in part 3, in a fraction of the time.

### 3. Young's method (gaps 1 to 4)

*Session 6, slides 11 to 17.*

**The policy turns the state into a Markov chain (slide 11).** A household at $(a_i, z_m)$ saves
$a'_{im} = g(a_i, z_m)$ and draws $z_n$ with probability $\pi_{mn}$. If $a'_{im}$ were a grid point $a_{G_{im}}$,
the pair $(a, z)$ would be a Markov chain with $N_a N_z$ states, and row $(i, m)$ of its transition matrix would
have $\pi_{mn}$ in column $(G_{im}, n)$. The distribution then moves forward exactly as the income distribution
did on slide 3, $\boldsymbol{\lambda}_{t+1} = \mathbf{Q}_{\mathbf{g}}^\top \boldsymbol{\lambda}_t$, and the stationary
distribution solves $\boldsymbol{\lambda} = \mathbf{Q}_{\mathbf{g}}^\top \boldsymbol{\lambda}$.

**The lottery (slide 12).** The EGM policy lies between grid points, $a'_{im} \in [a_l, a_{l+1}]$. Young (2010)
splits the mass with the linear interpolation weights: $1 - w_{im}$ to $a_l$ and $w_{im}$ to $a_{l+1}$, with

$$
w_{im} = \frac{a'_{im} - a_l}{a_{l+1} - a_l},
$$

a lottery over two grid points whose mean is exactly $a'_{im}$. Rounding to the nearest grid point would create
absorbing states (the fixed points around $k^*$ in problem set 1). The lottery lets mass flow.

`lottery` locates $l$ for every $(i, m)$ at once (`np.searchsorted` on the whole array, kept inside
$0, \ldots, N_a - 2$ so that $l + 1$ exists) and computes the weights.

In [ ]:
def lottery(G, gri):
    # For every policy value a'_im: the index l of the grid point just below it, and the weight w_im on the point above
    na = len(gri.a)
    l = np.clip(np.searchsorted(gri.a, G, side="right") - 1, 0, na - 2)   # locate: a_l <= a'_im < a_{l+1}, l at most N_a - 2
    # Gap 1: the weight on a_{l+1}: the distance of G from a_l, divided by the spacing a_{l+1} - a_l (gri.a[l]
    #        is an N_a x N_z array, like G)
    w = ___
    w = np.clip(w, 0.0, 1.0)                    # exactly at the top of the grid, w would be 1 + 1e-16
    return l, w

In [ ]:
l0, w0_ = lottery(G0, gri)
print("weights inside [0, 1]:", (w0_ >= 0).all() and (w0_ <= 1).all())
# The lottery reproduces the policy, except where the richest households at a = 100 would save beyond the grid
# (two points; the lottery caps them at a_N, and part 3 checks that no mass ever gets there)
print("lottery mean equals the policy (capped at a_N):", np.allclose((1 - w0_) * gri.a[l0] + w0_ * gri.a[l0 + 1], np.minimum(G0, gri.a[-1])))
print("policy values above the top of the grid:", (G0 > gri.a[-1]).sum())
i17 = np.argmin(np.abs(gri.a - 17.0))           # the example of slide 12: a = 17.0, income z_6
print(f"a = {gri.a[i17]:.2f}, z_6: a' = {G0[i17, 5]:.2f} lies between {gri.a[l0[i17, 5]]:.2f} and {gri.a[l0[i17, 5] + 1]:.2f}, w = {w0_[i17, 5]:.2f}")

**Check (slide 12):** at $a = 17.0$ with income $z_6$ the household saves $a' = 17.56$, between the grid points
$17.40$ and $17.82$, so $w = 0.40$. Two policy values sit above $a_N = 100$ (the two richest income states at the
top of the grid). The lottery caps them at $a_N$, which is harmless as long as no mass arrives there (check 3 below).

**Where the mass of one cell goes (slide 13, gap 2).** Take the cell $(a_i, z_m)$ of the example. Its mass
splits $2 N_z$ ways: for every income state $z_n$ tomorrow, a share $\pi_{mn} (1 - w_{im})$ lands on $(a_l, z_n)$
and a share $\pi_{mn} w_{im}$ on $(a_{l+1}, z_n)$. These $2 N_z$ numbers are the nonzero entries of row $(i, m)$ of
$\mathbf{Q}_{\mathbf{g}}$. Two things to check by hand: the shares add up to one (all the mass goes somewhere),
and the mean of tomorrow's assets under these shares is the policy value $a'_{im}$ itself (the lottery is fair).

In [ ]:
# The row of Q for the household of the example, written out: destination, share, as on slide 13
i, m = i17, 5                                   # the cell (a_i, z_m) = (17.0, z_6)
l_im, w_im = l0[i, m], w0_[i, m]
rows = []
for n in range(len(gri.z)):                     # income tomorrow
    # Gap 2: the share landing on (a_l, z_n): the income probability pi_mn times the lottery weight on the
    #        left neighbour, 1 - w_im
    rows.append((gri.a[l_im], gri.z[n], ___))
    rows.append((gri.a[l_im + 1], gri.z[n], gri.Pi[m, n] * w_im))   # the share landing on (a_{l+1}, z_n)
print(f"mass leaving (a, z) = ({gri.a[i]:.2f}, z_6), policy a' = {G0[i, m]:.2f}, lottery between {gri.a[l_im]:.2f} and {gri.a[l_im + 1]:.2f} with w = {w_im:.2f}")
print("   a' tomorrow   z tomorrow   share")
for a_to, z_to, share in rows:
    if share > 1e-4:                            # the chain never jumps more than two income states in practice, skip the tiny ones
        print(f"   {a_to:9.2f}   {z_to:9.2f}   {share:.4f}")
print(f"shares add up to {sum(s for _, _, s in rows):.6f}, mean of a' tomorrow {sum(a_to * s for a_to, _, s in rows):.4f} (the policy says {G0[i, m]:.4f})")

**Check (slide 13):** the largest shares go to $z_6$ tomorrow ($\pi_{66} = 0.73$ split $0.44$ and $0.29$), the
shares add up to one, and the mean of $a'$ tomorrow is $17.56$, the policy value. That is the whole of Young's
method, applied to one cell. The policy chain does it for all $1{,}400$ cells at once.

**The policy chain (slides 13 to 15, gap 3).** $\mathbf{Q}_{\mathbf{g}}$ has $N_a N_z = 1{,}400$ rows and columns but
only $2 N_z = 14$ nonzero entries per row, the shares of the previous cell: $\pi_{mn} (1 - w_{im})$ in column
$(l, n)$ and $\pi_{mn} w_{im}$ in column $(l + 1, n)$, for $n = 1, \ldots, N_z$. So it is built as a **sparse
matrix from triplets** (row, column, value), $19{,}600$ entries instead of $1{,}400^2 \approx 2$ million.

States are numbered by the table `idx`: `idx[i, m]` is the row (and column) of the state $(a_i, z_m)$, in the
order of Howard's step of week 5, $(i, m) \mapsto i + (m - 1) N_a$ (assets count fastest). The order is a
convention, but it must be the same for rows, columns, and the `reshape` that turns the vector
$\boldsymbol{\lambda}$ back into an $N_a \times N_z$ table. The loop runs over the income states $m$ today and $n$
tomorrow. Inside, all $N_a$ asset points are handled at once.

In [ ]:
def transition_matrix(G, gri):
    # Q[(i, m), (l, n)] = Pi[m, n] (1 - w_im) and Q[(i, m), (l + 1, n)] = Pi[m, n] w_im: the policy chain as a sparse matrix
    na, nz = len(gri.a), len(gri.z)
    idx = np.arange(na * nz).reshape(nz, na).T  # idx[i, m] = i + m N_a: the row of state (a_i, z_m), the order of Howard's step (week 5)
    l, w = lottery(G, gri)                      # the lottery of every (i, m)
    rows, cols, vals = [], [], []               # the triplets, collected column block by column block
    for m in range(nz):                         # income today
        for n in range(nz):                     # income tomorrow
            # mass 1 - w_im goes to (a_l, z_n) ...
            rows.append(idx[:, m]); cols.append(idx[l[:, m], n]); vals.append(gri.Pi[m, n] * (1 - w[:, m]))
            # ... and mass w_im to (a_{l+1}, z_n)
            # Gap 3: mirror the line above: the column of state (a_{l+1}, z_n), and the mass pi_mn w_im
            rows.append(idx[:, m]); cols.append(___); vals.append(___)
    return sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(na * nz, na * nz))

In [ ]:
t0 = time.perf_counter()
Q0 = transition_matrix(G0, gri)
print(f"Q built in {1000 * (time.perf_counter() - t0):.1f} ms: shape {Q0.shape}, {Q0.nnz} stored entries, "
      f"{100 * Q0.nnz / Q0.shape[0] ** 2:.2f} percent of the matrix")
rowsums = np.asarray(Q0.sum(axis=1)).ravel()
print(f"rows sum to one: largest deviation {np.max(np.abs(rowsums - 1)):.1e}")
idx0 = np.arange(mpar.na * mpar.nz).reshape(mpar.nz, mpar.na).T   # the same table, for looking up rows below
row = Q0[idx0[i17, 5]].toarray().ravel()        # the row of the example cell, as a dense vector
print(f"row of (17.0, z_6): {np.count_nonzero(row)} nonzero entries, largest {row.max():.4f}, sum {row.sum():.6f}")

**Check (slides 13 and 15):** $1{,}400 \times 1{,}400$, about $19{,}600$ stored entries ($1$ percent of the matrix;
a few entries are stored zeros where $w_{im} = 0$ exactly, at the limit), every row sums to one up to rounding,
and the row of the example cell has the $14$ entries of the previous cell, the largest $0.44$. Note on gap 3: the
hint names the two gaps of that line together, since one is the mirror of the other.

**The stationary distribution (slides 14 and 16, gap 4).** Row $(i, m)$ says where the mass at $(a_i, z_m)$
goes. Tomorrow's mass at a cell $(l, n)$ is what **arrives** there, from every cell that sends some: a column sum
weighted by today's distribution, $\lambda_{ln, t+1} = \sum_{(i, m)} Q_{(i, m), (l, n)} \lambda_{im, t}$, in one
line $\boldsymbol{\lambda}_{t+1} = \mathbf{Q}_{\mathbf{g}}^\top \boldsymbol{\lambda}_t$ (slide 14). Distributions move
with the **transpose**. `Q @ lam` would take expectations, the Howard step of week 5, and converge to a constant
vector. Start from the uniform distribution, or from a distribution passed in `lam0` (next week), and iterate until
the largest change is below $10^{-12}$.

In [ ]:
def stationary_dist(Q, mpar, gri, lam0=None):
    # Iterate lam_{n+1} = Q^T lam_n from the uniform distribution (or from lam0). Returns lam as an N_a x N_z table, and the count
    n = Q.shape[0]
    lam = np.full(n, 1 / n) if lam0 is None else lam0.T.ravel()   # the start, as a vector in the order of idx
    QT = Q.T.tocsr()                            # the transpose once, in the fast row format
    for it in range(1, mpar.maxiter + 1):
        # Gap 4: one step forward: the transpose of Q times the current distribution
        lam_new = ___
        dist = np.max(np.abs(lam_new - lam))    # largest change
        lam = lam_new
        if dist < mpar.crit_dist:
            break
    return lam.reshape(len(gri.z), len(gri.a)).T, it   # back to the (a, z) table, the same order as idx

In [ ]:
t0 = time.perf_counter()
lam0, itl = stationary_dist(Q0, mpar, gri)
t_young = time.perf_counter() - t0
print(f"Young's iteration: {itl} iterations to 1e-12, {1000 * t_young:.0f} ms")
# One step by hand, for one cell: the mass arriving at the example cell (17.0, z_6) is the weighted column sum of slide 14
lam_vec = lam0.T.ravel()                        # lam as a vector in the order of idx
col = Q0[:, idx0[i17, 5]].toarray().ravel()     # column (17.0, z_6) of Q: who sends mass there, and how much
print(f"mass arriving at (17.0, z_6): {col @ lam_vec:.3e} from {np.count_nonzero(col)} cells, against lam there {lam0[i17, 5]:.3e} (equal: stationary)")

**Aggregates and checks (slide 16).** Mean assets are $A = \boldsymbol{\lambda}^\top \mathbf{a} = \sum_{i, m} \lambda_{im} a_i$.
Then three checks before you trust a distribution: mass one, the marginal over assets equals
$\boldsymbol{\lambda}^z$ (a cheap test of the column index), and nothing at the top of the grid. The simulation
of part 2 is the fourth, external check.

In [ ]:
def mean_assets(lam, gri):
    # A = sum_{i, m} lam_im a_i, with lam as an N_a x N_z table
    return (lam * gri.a[:, None]).sum()

In [ ]:
A0 = mean_assets(lam0, gri)
lam_a0 = lam0.sum(axis=1)                       # the marginal distribution of assets
print(f"mean assets {A0:.4f} (simulation: {a_sim.mean():.4f})")
print(f"share at the limit {lam0[0].sum():.4f} (simulation: {(a_sim <= gri.a[0] + 1e-10).mean():.4f})")
print(f"check 1, mass: {lam0.sum():.12f}")
print(f"check 2, marginal over assets against lam_z: {np.max(np.abs(lam0.sum(axis=0) - gri.lam_z)):.1e}")
print(f"check 3, mass at the top of the grid: {lam0[-1].sum():.1e}")
print("mean assets by income state:", np.round((lam0 * gri.a[:, None]).sum(axis=0) / lam0.sum(axis=0), 1))
cdf0 = np.cumsum(lam_a0)
print(f"median {gri.a[np.searchsorted(cdf0, 0.5)]:.1f}, share below one year of mean income: {cdf0[np.searchsorted(gri.a, gri.lam_z @ gri.z) - 1]:.2f}")

**Check (slides 16 and 17):** $1{,}267$ iterations in a few tens of milliseconds, mean assets $8.517$ against about
$8.50$ in the simulation, $1.6$ percent at the limit, mass $1$ to twelve digits, the marginal over assets equal
to $\boldsymbol{\lambda}^z$ to $10^{-13}$, $10^{-9}$ at $a = 100$. Mean assets by income state rise from $4.2$ at
$z_1$ to $17.3$ at $z_7$: the precautionary motive at work. Median $6.3$, $14$ percent below one year of mean income.

In [ ]:
# Nothing to fill in: the simulated cross section against Young's distribution (session 6, slide 17).
# Young's mass per grid point is turned into a density by dividing by the width of the cell around each point.
def density(lam_a, agrid):
    mids = np.r_[agrid[0], (agrid[:-1] + agrid[1:]) / 2, agrid[-1]]   # cell boundaries: midpoints between neighbours
    return lam_a / np.diff(mids)                # mass per unit of assets


fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.5, 1]})
edges = np.linspace(0, 30, 61)
inside = a_sim > gri.a[0] + 1e-10               # the histogram without the spike at the limit
ax[0].hist(a_sim[inside], bins=edges, weights=np.full(inside.sum(), 1 / N / (edges[1] - edges[0])), alpha=0.6,
           label=f"simulated cross section, N = {N:,}, T = {T:,}")
ax[0].plot(gri.a[1:], density(lam_a0, gri.a)[1:], "k-", lw=1.2, label="Young's method, mass per unit of assets")
ax[0].set(xlabel="assets $a$", ylabel="density", xlim=(0, 30), title="Same distribution")
ax[0].legend(fontsize=7)
ax[1].plot(np.arange(1, T + 1), means, lw=1.2, label="mean assets in the simulation")
ax[1].axhline(A0, color="k", lw=0.8, ls="--", label="Young's method")
ax[1].set(xscale="log", xlabel="period $t$", ylabel="mean assets", title="The burn in")
ax[1].legend(fontsize=7, loc="lower right")
plt.tight_layout()
plt.show()

**To discuss (slide 17).** The histogram sits on Young's density, and the two means differ by less than one
standard error of the simulation. What Young's method assumes: mass sits on grid points (the grid sets the
resolution), the mean is exact, the variance is slightly too large by the lottery's spread. What it needs: a grid
for the endogenous state and a finite chain for the exogenous one. Simulate anyway when the state space is too
large for the matrix, when the question is about individual histories, or once, as a check of a new
$\mathbf{Q}_{\mathbf{g}}$, as here.